In [1]:
from src.feature_engineering import feature_engineering

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import re


In [2]:
# Load engineered data
X_train, X_val, X_test, y_train, y_val, y_test, scaler = feature_engineering()

C:\Users\taula\OneDrive\Desktop\Sem7\DSPRO\hospital-readmission-predictor\src\data_cleaning.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_data['max_glu_serum'].fillna(0, inplace=True)
C:\Users\taula\OneDrive\Desktop\Sem7\DSPRO\hospital-readmission-predictor\src\data_cleaning.py:32: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object o

## Feature selection

In [3]:

# Group one-hot columns by their base category
groups = {}
for col in X_train.columns:
    base = re.sub(r'_[^_]+$', '', col)  # removes the last suffix after underscore
    groups.setdefault(base, []).append(col)

# Initial feature importance selection
selector_model = RandomForestClassifier(n_estimators=100, random_state=42)
selector_model.fit(X_train, y_train)

importances = pd.Series(selector_model.feature_importances_, index=X_train.columns)
top_features = importances.nlargest(25).index

# Enforce full group inclusion
final_features = set()
for g, cols in groups.items():
    if any(c in top_features for c in cols):
        final_features.update(cols)

X_train = X_train[list(final_features)]
X_val = X_val[list(final_features)]
X_test = X_test[list(final_features)]
print("Final grouped features:", final_features)


Final grouped features: {'diag_3_category_Diseases of the nervous system and sense organs', 'medical_specialty_Nephrology', 'diag_2_category_Diseases of the digestive system', 'diag_2_category_Diseases of the musculoskeletal system and connective tissue', 'A1Cresult', 'age_[80-90)', 'stay_length_cat', 'diag_3_category_Neoplasms', 'age_[40-50)', 'glyburide', 'medical_specialty_Urology', 'discharge_disposition_Discharged/transferred to a nursing facility certified under Medicaid but not certified under Medicare.', 'age_[90-100)', 'age_[20-30)', 'discharge_disposition_Left AMA', 'medical_specialty_InternalMedicine', 'diag_1_category_Injury and poisoning', 'medical_specialty_Speech', 'medical_specialty_Gynecology', 'diag_1_category_Endocrine, nutritional and metabolic diseases, and immunity disorders', 'discharge_disposition_Discharged/transferred/referred another institution for outpatient services', 'discharge_disposition_Discharged/transferred to home under care of Home IV provider', 'd

In [5]:
for df in [X_train, X_val, X_test]:
    df.columns = [re.sub(r'[^A-Za-z0-9_]+', '_', col) for col in df.columns]

# Set MLflow tracking URI and experiment
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("model-comparison")

# Define models with their configurations
models = {
    "Logistic_Regression": {
        "model": LogisticRegression(),
        "log_function": mlflow.sklearn.log_model
    },
    "XGBoost": {
        "model": XGBClassifier(),
        "log_function": mlflow.xgboost.log_model
    },
    "Decision_Tree": {
        "model": DecisionTreeClassifier(random_state=0),
        "log_function": mlflow.sklearn.log_model
    },
    "Random_Forest": {
        "model": RandomForestClassifier(class_weight='balanced'),
        "log_function": mlflow.sklearn.log_model
    }
}

# Dictionary to store results
results = {}

# Train and track each model
for model_name, model_config in models.items():
    print(f"\nTraining {model_name}...")

    with mlflow.start_run(run_name=model_name):
        # Get model and logging function
        model = model_config["model"]
        log_function = model_config["log_function"]

        # Log model parameters
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("train_size", len(X_train))
        mlflow.log_param("val_size", len(X_val))

        # Log specific model parameters
        for param_name, param_value in model.get_params().items():
            mlflow.log_param(param_name, param_value)

        # Train model
        model.fit(X_train, y_train)

        # Make predictions
        predictions = model.predict(X_val)

        # Calculate metrics
        accuracy = accuracy_score(y_val, predictions)
        f1 = f1_score(y_val, predictions, average='weighted')
        precision = precision_score(y_val, predictions, average='weighted')
        recall = recall_score(y_val, predictions, average='weighted')

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)

        # Log the model
        log_function(model, model_name.lower())

        # Store results
        results[model_name] = accuracy

        # Log tags for easy filtering
        mlflow.set_tag("stage", "validation")
        mlflow.set_tag("dataset", "current_dataset")

        print(f"{model_name} - Accuracy: {accuracy:.4f}, F1: {f1:.4f}")

# Create results dataframe
df_results = pd.DataFrame({
    "LR": [results["Logistic_Regression"]],
    "XGB": [results["XGBoost"]],
    "DT": [results["Decision_Tree"]],
    "RF": [results["Random_Forest"]]
})

print("\n" + "="*50)
print("RESULTS SUMMARY")
print("="*50)
print(df_results)

# Log the comparison results as an artifact
with mlflow.start_run(run_name="comparison_summary"):
    mlflow.log_param("comparison_type", "all_models")

    # Save and log the results dataframe
    df_results.to_csv("model_comparison.csv", index=False)
    mlflow.log_artifact("model_comparison.csv")

    # Log the best model info
    best_model = max(results, key=results.get)
    best_score = results[best_model]
    mlflow.log_param("best_model", best_model)
    mlflow.log_metric("best_accuracy", best_score)

    print(f"\nBest Model: {best_model} with accuracy: {best_score:.4f}")


2026/01/11 15:56:49 INFO mlflow.tracking.fluent: Experiment with name 'model-comparison' does not exist. Creating a new experiment.



Training Logistic_Regression...


C:\Users\taula\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/01/11 15:56:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/11 15:56:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logistic_Regression - Accuracy: 0.6348, F1: 0.6284
🏃 View run Logistic_Regression at: http://localhost:5000/#/experiments/860725067918785717/runs/75f3b96fd7794c87a74dea2bb4d40e85
🧪 View experiment at: http://localhost:5000/#/experiments/860725067918785717

Training XGBoost...


2026/01/11 15:56:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/11 15:57:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


XGBoost - Accuracy: 0.6418, F1: 0.6386
🏃 View run XGBoost at: http://localhost:5000/#/experiments/860725067918785717/runs/aab8572162444c10bf28c23a89c4d753
🧪 View experiment at: http://localhost:5000/#/experiments/860725067918785717

Training Decision_Tree...


2026/01/11 15:57:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/11 15:57:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Decision_Tree - Accuracy: 0.5573, F1: 0.5573
🏃 View run Decision_Tree at: http://localhost:5000/#/experiments/860725067918785717/runs/40e0f9e489d74696843944179a05a9b6
🧪 View experiment at: http://localhost:5000/#/experiments/860725067918785717

Training Random_Forest...


2026/01/11 15:57:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/11 15:57:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Random_Forest - Accuracy: 0.6305, F1: 0.6272
🏃 View run Random_Forest at: http://localhost:5000/#/experiments/860725067918785717/runs/3627220958ab47a780f42ba74b51f5e9
🧪 View experiment at: http://localhost:5000/#/experiments/860725067918785717

RESULTS SUMMARY
         LR       XGB        DT        RF
0  0.634755  0.641832  0.557303  0.630529

Best Model: XGBoost with accuracy: 0.6418
🏃 View run comparison_summary at: http://localhost:5000/#/experiments/860725067918785717/runs/9218000b7cbc447984f7729731a09061
🧪 View experiment at: http://localhost:5000/#/experiments/860725067918785717


In [ ]:
df